# Run inference for Qwen3TTS

This notebook demonstrates how to:
1. Initialize the Qwen3TTS model with dummy weights
2. Run inference with random embeddings to verify the model works


In [ ]:
import os
os.environ["VLLM_ATTENTION_BACKEND"] = "TRITON_ATTN"


In [ ]:
import torch
from pathlib import Path
from vllm import SamplingParams
from vllm.engine.arg_utils import AsyncEngineArgs
from vllm.v1.engine.async_llm import AsyncLLM

print("Imports successful")


In [ ]:
# Load vLLM engine with dummy model
type_str = "bfloat16"
torch_type = getattr(torch, type_str)

max_len = 256
config_path = Path("dummy_qwen3_tts_model")
engine_args = AsyncEngineArgs(
    model=str(config_path.absolute()),
    dtype=type_str,
    max_model_len=max_len,
    max_num_batched_tokens=max_len,
    gpu_memory_utilization=0.6,
    skip_tokenizer_init=True,  # Skip tokenizer since we're using custom inputs
    enable_prefix_caching=False,
    #load_format="dummy",  # Use dummy loader for random weights
    trust_remote_code=True,
    #enforce_eager=True,
    input_coalesce_timeout_ms=5,
    shm_decode=True,
)

print("Initializing engine...")
engine = AsyncLLM.from_engine_args(engine_args)
sampling_params = SamplingParams(max_tokens=max_len, skip_sampling=False)

print("Engine initialized successfully")


In [ ]:
# load prefill emb that contains text to generate and speaker info

prefill_data = torch.load("/home/vklimkov/workspace/vllm/vllm/dummy_qwen3_tts_model/prefill_input.pt")
prefill_emb = prefill_data["talker_input_embeds"][0].contiguous().cpu()
print(prefill_emb.shape)

In [ ]:

request_id = "test_request_1"


# Prepare inputs for prefill stage
prompt_len = prefill_emb.shape[0]

#prefill_emb = torch.randn_like(prefill_emb)
inputs = {
    # Dummy token IDs (required by vLLM, but we use custom_inputs for actual data)
    "prompt_token_ids": [0] * prompt_len,
    # Custom inputs for Qwen3TTS
    "custom_inputs": {
        "combined_embeddings": prefill_emb,
    }
}

print(f"Starting generation with request_id: {request_id}")
print(f"Prompt length: {prompt_len}")



# 1. Submit the request.  shm_decode=True means add_request()
#    internally creates and registers a SHM channel.
queue = await engine.add_request(request_id, inputs, sampling_params)
prefill_output = await queue.get()
next_input = prefill_output.outputs[0].custom_outputs["next_input_embeddings"][-1:, :]
next_tokens = prefill_output.outputs[0].custom_outputs["codes"][-1:]

generated_codecs = []


for i in range(max_len):
    generated_codecs.append(next_tokens)
    outputs = engine.decode_step_shm(request_id, custom_inputs={"combined_embeddings": next_input})

    next_input = outputs["next_input_embeddings"][-1:, :]
    next_tokens = outputs["codes"][-1:].clone()

    if i >= max_len - prompt_len:
        await engine.abort(request_id)
        break

    codec_eos_token_id = 2150  # from talker_config.codec_eos_token_id
    if next_tokens[0, 0].item() == codec_eos_token_id:
        print(f"EOS token detected at step {i + 1}, stopping generation.")
        await engine.abort(request_id)
        break

In [ ]:
arr = torch.cat(generated_codecs, dim=0)

import matplotlib.pyplot as plt

plt.imshow(arr.cpu().numpy().T, aspect='auto')
plt.colorbar()
plt.show()

In [ ]:
print(arr.shape)
print(torch.min(arr), torch.max(arr))
torch.save(arr, "/home/vklimkov/workspace/qwen3_tts/Qwen3-TTS/vllm_pred_tokens.pt")